# 第66课 · 从声波到文字的完整链路——ASR 系统全览：端到端（end-to-end） vs 经典流水线

**目标**：先把 ASR 的整条链路看清：特征 → 编码器 → 解码器 → 文字。

> **地图课**：先画框图，再扫一遍 L67–L75；本课只抓整体，不在单点上深挖（CTC / Whisper 后面单独讲）。

🔗 **Aurora 连接**：`aurora.audio.mel` 实现了 ASR 的特征提取层（log-Mel 频谱（log-Mel spectrogram））；`aurora.speech`（本月新建）以后会用来放 CTC/Attention 编解码器。

← **上一课**　[L65 · 训练可视化](../6_deep_learning/L65_visual_training.ipynb)

> 上节课学习了 **训练可视化**：Loss / Acc 曲线，梯度范数，权重分布直方图。  
> 本课将探讨 **ASR 系统全览**。

## 本课剧情：Siri 怎么把你说的话变成文字？

你说一句话，Siri 在 1 秒内把它变成文字。中间到底发生了什么？

**问题的难点**：
1. 声音是连续的波形，文字是离散的 token
2. 同一个词，不同人说的时间长度不一样（"hello"可能 0.3 秒也可能 0.7 秒）
3. 不同语境同一词的发音差异巨大（快读、口音、噪声）

**端到端 ASR 的三步链条**：
```
波形 (16kHz) → [声学编码器] → 特征序列 → [语言解码器] → token序列 → 文字
```

**传统 vs 端到端的关键区别**：

| | 传统 HMM-GMM | 端到端 (Whisper) |
|---|---|---|
| 声学模型（acoustic model，AM） | GMM，每帧独立建模 | Transformer encoder，全局注意力 |
| 语言模型（language model，LM） | 独立训练，接缝处有误差 | 端到端联合优化 |
| 解码 | Viterbi，依赖发音词典 | Beam search on token logits |
| 训练数据 | 对齐文本+音频（贵） | 任意语音+转写（便宜）|

**评估指标**：WER（Word Error Rate）= 把假设文本改成参考文本所需的最少词操作数 / 参考词数：
```
WER = (替换S + 删除D + 插入I) / N（参考词数）
```

本节任务：实现 `compute_wer(hypothesis, reference)` — 用 Levenshtein 动态规划（dynamic programming，DP）算 WER。

In [ ]:
import numpy as np

## 1. 端到端 ASR vs 传统 HMM-GMM

传统流水线（pipeline）把任务切成三段独立训练：**声学模型**（GMM 建模帧概率）→ **发音词典（pronunciation dictionary / lexicon）** → **语言模型**（N-gram）。误差无法端到端反传，每段单独调到最好，合起来不一定最好。

端到端系统（Whisper/wav2vec 2.0）直接学 P(文字 | 音频)：Transformer 编码器处理帧序列，解码器（或 CTC 头）输出 token 概率。一次梯度下降（gradient descent）同时优化全链路，数据量够大时效果碾压传统方案。

架构对比（简化）：
```
HMM-GMM: waveform → MFCC → GMM/HMM → 词典 → N-gram LM → text
Whisper:  waveform → log-Mel → Transformer Enc → Transformer Dec → text
```

### 补充：为什么叫「编码器」「解码器」？

上一段说"Transformer 编码器处理帧序列，解码器输出 token 概率"——这两个名字不是随便起的，而是精确描述了各自在做什么操作。

**一个类比：寄快递**

- 你要把一个易碎的花瓶寄给朋友。直接扔进纸箱会摔碎，所以你先把它**打包**——用泡沫纸把花瓶包成一个标准尺寸的包裹（不管花瓶本身形状多奇怪，包裹外形是统一的）。这个"打包"的过程就类似**编码（encode）**：把一个不规则、原始的东西，转换成一种更规整、更抽象的表示。
- 朋友收到包裹后，要把泡沫纸拆开，取出花瓶，才能真正"看懂"里面是什么。这个"拆包"的过程就类似**解码（decode）**：把压缩过的抽象表示，还原成人能理解的具体内容。

**放回 ASR 里，具体发生了什么？**

- **Encoder（编码器）**：输入是一段音频对应的 log-Mel 帧序列——不同的话说出来长度不一样（"hello"可能是 30 帧，也可能是 70 帧）。Encoder 用很多层 Transformer 处理这些帧，把每一帧都转换成统一维度的隐状态向量（比如 512 维）。**关键点**：不管原始音频有多长，每一帧输出的"表示"维度是固定的——这就是"编码"：把不规则的原始声学信号，压缩/转换成模型内部统一的、抽象的数字表示。
- **Decoder（解码器）**：输入是 encoder 产出的隐状态，外加自己已经写出来的文字。Decoder 一步一步地把这些抽象的隐状态"翻译"回人类能读懂的文字 token——这就是"解码"：把压缩过的抽象表示，逐步展开还原成具体、可读的结果。

**为什么不干脆叫"前半部分""后半部分"？**
因为这两部分做的是性质完全不同的操作——一个是"压缩/抽象化"（把具体的声音信号变成模型内部的数字表示），一个是"复原/具象化"（把数字表示还原成人能读的文字）。"编码/解码"这对名字精确地传达了这种操作上的对称性和方向性，而"前/后"只表达了顺序，丢掉了这层含义。这也是为什么几乎所有类似结构（压缩/解压缩、加密/解密、编码器-解码器架构）都用这一对术语。

### ⚠️ 快速概念说明：HMM 和 GMM 是什么？

**本段是为第一次接触这些术语的学生写的。**如果你已熟悉这些概念，可跳过。

语音识别说"GMM 建模帧概率"、"HMM 状态后验概率（posterior probability）"，这两个名字很陌生。它们是什么？

**GMM（Gaussian Mixture Model，高斯混合模型）**：
- 简单说，就是把每一**帧音频**用若干个"钟形曲线"（高斯分布）的加权和来表示
- 帧的作用是什么？把连续的波形分成小窗口（10ms一帧），每帧看作一个"小截图"
- 对于"啊"这个音素，第1帧可能是"分布A"，第2帧是"分布B"，……
- **问题**：GMM 只关注**单帧**，不考虑帧与帧之间的关联（"啊"的动态变化）

**HMM（Hidden Markov Model，隐马尔可夫模型）**：
- 把 GMM 的"帧概率"用一个**状态机**串起来
- 状态机的逻辑是：上一帧是"发音起始"→ 当前帧是"发音中期" → 下一帧是"发音结束"
- HMM 让系统记住：某个音素的发音通常有一个**时间顺序**，而不是每帧独立的
- **优点**：比 GMM 多了"时序性"；**问题**：实现复杂，需要手工设计状态转移规则

**为什么这些东西过时了？**
传统方案的痛点是：GMM 和 HMM 分开训练，组合时容易出现"缝隙"（一个组件优化了，但整体不优；另一个跟着变差）。现代端到端系统直接学 P(文字|音频)，用一个大的梯度下降把所有参数同时优化，避免了这个问题。

---

In [ ]:
# 演示：两种路径的「接口」差异
# 传统方案输出每帧的 HMM 状态后验概率；端到端直接输出 token logits

T, n_mels = 300, 80   # 300 帧，80维 log-Mel
n_vocab   = 51865     # Whisper 词表大小

# 传统：每帧 → GMM 状态概率（假设 3000 个 HMM 状态）
n_hmm_states = 3000
hmm_output_shape = (T, n_hmm_states)

# 端到端：编码器输出序列 → 解码器输出 token 概率
enc_output_shape = (T // 2, 512)   # 2× 下采样后，d_model=512
dec_output_shape = (n_vocab,)       # 每步预测一个 token

print(f'HMM-GMM 每帧输出维度: {hmm_output_shape}')
print(f'Whisper 编码器输出形状: {enc_output_shape}')
print(f'Whisper 解码器每步输出维度: {dec_output_shape}')

### 补充：什么是"下采样（downsampling）"（downsampling）和 stride？

上面代码提到"2× 下采样"，这是什么意思？

**下采样的直观比喻**：
- 想象一部 30fps 的视频（每秒30帧），现在"每隔1帧取1次"，变成15fps
- 或者"每隔2帧取1次"，变成 7.5fps
- 空间上呢？一张 1000×1000 的高分辨率图片，"每隔1个像素取1个"，变成 500×500 的缩小图

**在卷积神经网络中**：
- `stride=1` 表示滑动窗口每次移动1个位置，所有位置都会被处理
- `stride=2` 表示滑动窗口每次移动2个位置，相当于"隔一个、取一个"
- 结果就是输出序列长度减半

**为什么 ASR 编码器要下采样？**
1. **计算省钱**：300 帧变成 150 帧，后续处理量减少，模型参数减少，速度变快
2. **抓大放小**：细微的时间差（比如 2 帧 = 20ms 的差别）对识别词往往不重要；下采样后，"大的时序模式"（比如 200ms 的声调起伏）反而更清晰，模型更容易学

**Whisper 的设计**：用 2 层卷积，第一层 stride=1，第二层 stride=2，总共 2× 下采样。这个系数是经验选择（过度下采样会丢失细节，完全不采样又太慢）。

---

### 1b. 三大模块的输入/输出形状与参数量

端到端 ASR 可以拆解为三个可复用模块，先搞懂每个模块的接口，才能把它们拼成一个系统：

| 模块 | 输入 | 输出 | 典型参数量（Whisper-small） |
|---|---|---|---|
| 声学编码器（AM/Encoder） |  log-Mel 频谱 |  隐状态序列 | ≈ 88 M |
| 语言解码器（LM/Decoder） | 编码器输出 + 已生成 token |  下一词 logits | ≈ 154 M |
| 束搜索解码器（Beam Decoder） | logits 序列 | token id 列表 → 文字 | 无可训练参数 |

传统 HMM-GMM 中，三段**分开训练**，误差无法端到端反传；
Whisper 等 E2E 模型用统一的交叉熵损失对齐三段，显著降低工程复杂度。

In [ ]:
# 演示：AM / LM / Decoder 的尺寸变换
B, T, n_mels = 2, 300, 80
encoder_d = 512
vocab_size = 51865

# Step 1 — 声学编码器：T 帧 → T/2 帧（Whisper 用 2 层 conv，其中第二层 stride=2 → 2× 下采样）
am_out_shape = (B, T // 2, encoder_d)
print(f"AM  输出: {am_out_shape}")

# Step 2 — 语言解码器：每步生成 1 个 token（自回归）
lm_out_shape = (B, vocab_size)
print(f"LM  输出（每步）: {lm_out_shape}")

# Step 3 — 束搜索：保留 beam_size 条候选路径
beam_size = 5
beam_candidates = beam_size
print(f"束搜索: 每步保留 {beam_candidates} 条候选")

# AM / LM 参数量粗估（Whisper-small，官方标称总参数 244M）
am_params  = 88e6
lm_params  = 154e6
print(f"Whisper-small: AM ≈ {am_params/1e6:.0f}M 参数，LM ≈ {lm_params/1e6:.0f}M 参数")
print(f"总参数量 ≈ {(am_params + lm_params)/1e6:.0f}M")


### 补充：什么是「自回归」（autoregressive）？Beam Search 为什么有效？

上面代码里出现了两个新词——"自回归"和"束搜索（beam search）"。它们都在回答同一个问题：**decoder 一步步生成 token 时，到底是怎么"一步步"的？**

**自回归（autoregressive）是什么？**

拆开这个词：`auto` = 自己，`regress`（这里取广义含义）= 推算/预测。合起来：**自回归 = 根据自己之前已经生成的结果，来推算下一步的结果。**

类比手机输入法：你打"我今天很"，输入法会根据你已经打出的这几个字，预测下一个字最可能是"开心""累""忙"。它预测下一步，靠的是"看自己已经写出来的东西"——这就是自回归。

放回 ASR 解码器里，具体发生的事情是：
```
第1步：只有 encoder 的隐状态可用（还没生成任何词）→ 生成 token_1
第2步：encoder 隐状态 + token_1（自己刚生成的）→ 生成 token_2
第3步：encoder 隐状态 + token_1, token_2（自己刚生成的两个）→ 生成 token_3
...
```
每一步的输入，都包含了"自己之前所有步骤的输出"——这正是代码注释里"每步生成 1 个 token（自回归）"这句话的准确含义。（这里的"回归"不是统计学里的线性回归，是"预测/推算"的广义用法，不要和 L23-25 的梯度下降/回归模型混淆。）

---

**Beam Search 为什么有效？和"每步只选最高分"有什么不同？**

既然是自回归、一步步生成，那最简单的想法是：每一步都只选"当前看起来概率最高的那个词"，这叫**贪心搜索（greedy search）**。但贪心搜索有个致命问题——类比走迷宫：如果你每一步都只选"眼前看起来最近的岔路"，一旦走错一步，后面所有路都建立在这个错误之上，没法回头重选，很容易走进死胡同（这在算法里叫陷入**局部最优**，不是全局最优）。

**Beam Search 的思路**：不要只留 1 条路，而是**同时保留 `beam_size` 条（比如 5 条）得分最高的候选序列**，一起往前走：
1. 每一步，对每一条当前保留的候选序列，都尝试所有可能的下一个词，算出新的累计得分
2. 把所有"候选序列 × 词表所有词"的组合放在一起比较，只保留得分最高的 `beam_size` 条，扔掉其余的
3. 重复，直到生成结束符

因为同时探索了多条路，即使某一条路的某一步不是"当前看起来最优"，只要它整体得分更高，也有机会被保留下来、最终胜出——这就是 beam search 比贪心搜索更容易找到全局更优序列的原因。

**为什么叫"束"（beam）？** 想象手电筒射出的光束（beam）——不是一条细线，而是同时照亮一片区域的多条路径。Beam search 同时"照亮"（探索）多条有希望的候选路径，而不是死盯着一条。

**和 Viterbi 的区别**（回应表格里"解码：Viterbi vs Beam search"这一行）：Viterbi 是**精确算法**——在 HMM 这种状态数有限、结构固定的模型里，它能保证找到全局最优路径。Beam search 是**近似算法**——不保证一定是全局最优，但计算量可控：Transformer 解码器的"状态空间"是任意长度的 token 序列，词表也有几万个词，穷举所有可能根本不现实，beam search 用"保留有限的几条候选"来换取可行的计算量。

In [ ]:
# 演示：贪心搜索 vs Beam Search —— 为什么保留多条候选能找到更好的整体结果
#
# 场景设计：故意让"贪心搜索"在第一步就走错——它选了当下分数更高的 'A'，
# 但 'A' 之后的路都很差；而暂时看起来分数较低的 'B'，走下去反而有条高分的路。
# scores_table[已生成的序列] = {下一个候选词: 该词的得分}
scores_table = {
    (): {'A': 0.6, 'B': 0.4},        # 第1步：A 暂时领先
    ('A',): {'A': 0.1, 'B': 0.1},    # 选了 A 之后，两条路都很差（死胡同）
    ('B',): {'A': 0.9, 'B': 0.1},    # 选了 B 之后，接 A 是个很好的选择
}

def greedy_search(steps=2):
    """每一步只保留 1 条：当前分数最高的候选。"""
    seq, total_score = (), 0.0
    for _ in range(steps):
        next_scores = scores_table[seq]
        best_word = max(next_scores, key=next_scores.get)
        total_score += next_scores[best_word]
        seq = seq + (best_word,)
    return seq, total_score

def beam_search(beam_size=2, steps=2):
    """每一步保留 beam_size 条累计得分最高的候选序列。"""
    beams = [((), 0.0)]  # [(已生成序列, 累计得分), ...]
    for _ in range(steps):
        candidates = []
        for seq, score in beams:
            for word, s in scores_table.get(seq, {}).items():
                candidates.append((seq + (word,), score + s))
        candidates.sort(key=lambda x: -x[1])
        beams = candidates[:beam_size]   # 只留得分最高的 beam_size 条
    return max(beams, key=lambda x: x[1])

g_seq, g_score = greedy_search()
b_seq, b_score = beam_search(beam_size=2)

print(f"贪心搜索 (每步只留1条)  结果: {g_seq}, 总分={g_score:.2f}")
print(f"Beam Search (beam_size=2) 结果: {b_seq}, 总分={b_score:.2f}")
print()
print("贪心第一步看到 A(0.6) > B(0.4)，选了 A —— 但 A 之后的路都很差 (0.1)。")
print("Beam Search 同时保留了 A 和 B 这两条候选，才发现 B 之后接 A 能拿到 0.9 的高分，")
print("最终找到了总分更高的序列。这就是 beam_size > 1 比贪心 (相当于 beam_size = 1) 更强的原因。")

## 2. 特征：80-dim log-Mel 频谱

Whisper 的标准输入是 **80维 log-Mel 频谱**，帧参数与 L43-L47 完全一致：
```
采样率   sr = 16000 Hz
窗长     n_fft  = 400  样本  （25 ms）
帧移     hop    = 160  样本  （10 ms）
Mel bins n_mels = 80
```

处理流程：`stft → |·|² → mel_filterbank · power → log(max(·, 1e-10))`。结果形状为 `(80, T_frames)`，T_frames ≈ 总样本数 / hop。

为什么 Mel 而非线性频率？人耳对低频分辨率更高，Mel 尺度就是照着人耳这个特点设计的，让模型更容易学到音素（phoneme）边界。

In [ ]:
# 演示：STFT 帧数公式推导 + log-Mel 频谱形状

# ===== STFT 帧数公式：为什么是 1 + (n_samples - n_fft) // hop？=====
# 
# 直观理解（从几何角度）：
# 想象一条时间轴，长度 = n_samples 个样本点（比如 48000 个）。
# 现在用一个"窗口"（n_fft 个样本宽），从左往右滑动。
# 每次滑动 hop 个样本（比如 160），计算一次 FFT。
#
# 关键问题：从位置 0 开始，能滑多少次？
# - 第1次窗口：位置 0 到 399（n_fft-1）
# - 第2次窗口：位置 160 到 559
# - 第3次窗口：位置 320 到 719
# - ……
# - 最后一次窗口：位置必须满足 start + n_fft - 1 < n_samples
#   即 start < n_samples - n_fft + 1
#   即 start ∈ [0, n_samples - n_fft]
#
# 如果每次 start 增加 hop，从 0 开始，最大能到 n_samples - n_fft：
#   能滑的次数 = floor((n_samples - n_fft) / hop) + 1
#
# 这就是公式 1 + (n_samples - n_fft) // hop

sr     = 16000
n_fft  = 400
hop    = 160
n_mels = 80
duration_s = 3.0

n_samples  = int(sr * duration_s)
n_frames   = 1 + (n_samples - n_fft) // hop

print(f'=== STFT 帧数推导 ===')
print(f'音频长度: {duration_s} s  →  {n_samples} 样本')
print(f'窗长: n_fft = {n_fft} 样本 = {n_fft/sr*1000:.1f} ms')
print(f'帧移: hop = {hop} 样本 = {hop/sr*1000:.1f} ms')
print(f'公式: n_frames = 1 + ({n_samples} - {n_fft}) // {hop}')
print(f'     = 1 + {n_samples - n_fft} // {hop}')
print(f'     = 1 + {(n_samples - n_fft) // hop}')
print(f'     = {n_frames}')
print(f'\nlog-Mel 频谱形状: ({n_mels}, {n_frames})')
print(f'每帧覆盖时间: {n_fft/sr*1000:.1f} ms，相邻帧间隔: {hop/sr*1000:.1f} ms')

### 深入理解：log-Mel 特征提取的每一步

在讲帧数计算之前，先把"stft → |·|² → mel_filterbank · power → log(max(·, 1e-10))"这个流程拆开讲。

**第1步：STFT（短时傅里叶变换，Short-Time Fourier Transform）**
- **作用**：把波形分成短窗口（每个 25ms），对每个窗口做傅里叶变换
- **输入**：一维波形（时间序列）
- **输出**：二维数据，行=频率，列=时间帧（每帧一个频谱）
- **结果的值**：复数（每个复数 = 某个频率在某一帧的"强度+相位"）

**第2步：|·|²（取模平方）**
- **目的**：从复数提取"能量"，丢掉相位信息（因为相位对识别词的帮助不大）
- **具体做法**：如果 STFT 输出复数 `z = a + bi`，那么 `|z|² = a² + b²`
- **结果**：正实数，形状是 `(频率数, 帧数)`，比如 `(513, 300)`

**第3步：mel_filterbank · power（Mel 滤波器组）**
- **问题**：人耳对高频敏感度低，对低频敏感度高（你能听出男女声的低频差异，但听不出高频的细微差别）
- **解决**：用"Mel 滤波器组"，把 513 条频率轴压缩为 80 条（Mel 尺度）
  - Mel 公式：`mel = 2595 * log10(1 + hz/700)` （从频率Hz变成感知尺度 mel）
  - 低频部分（如 0-1000 Hz）被展开成很多条滤波器；高频部分（如 8000-16000 Hz）被压缩成少数几条
  - 这样模型在低频细节上有更多"分辨力"
- **计算方式**：矩阵乘法 `(80, 513) @ (513, 300) = (80, 300)`
- **结果**：80 维 log-Mel 频谱（还没取 log）

**第4步：log(max(·, 1e-10))（对数压缩 + 数值稳定）**
- **为什么取 log？**
  - 原始能量跨度很大（0 到几百），不利于神经网络学习（梯度会很不均衡）
  - 取 log 后，所有值被压缩到一个更小的范围；同时，两个很不同的数值（如 0.001 和 0.999）被映射到更接近的值，便于模型判别
  - 例如：`log(0.001) ≈ -6.9, log(0.999) ≈ -0.001`，两者相差 6.9；不取 log 时，两者相差 0.998，很容易被淹没
- **为什么是 max(·, 1e-10)？**
  - 能量有可能是 0（无声段），但 `log(0)` 无定义（趋向 -∞）
  - 加这个 max 操作，保证输入 ≥ 1e-10，这样 `log(1e-10) ≈ -23`，一个有限的大负数
  - 1e-10 这个阈值是经验选择，足够小（不会错误地提升接近 0 的真实能量），又足够大（避免浮点数下溢）
- **结果**：80 维 log-Mel 频谱，范围大约 -25 到 10，适合神经网络处理

---

## 3. WER — 词错率

WER（Word Error Rate）是 ASR 最核心的评估指标：
```
WER = (S + D + I) / N
```
- `N`：参考（正确）文本的总词数
- `S`（Substitution）：替换错误数——模型输出了错词
- `D`（Deletion）：删除错误数——漏说了某词
- `I`（Insertion）：插入错误数——多说了某词

WER 通过**编辑距离**（Levenshtein distance）计算，以词为单位而非字符。WER 可以 > 1.0（大量插入时）。

常见基线（英语）：
```
Whisper-large-v3    WER ≈ 2–5%   (LibriSpeech test-clean)
Whisper-base        WER ≈ 5–8%
传统 HMM-GMM        WER ≈ 8–15%
```

In [ ]:
# 演示：手工追踪一个 WER 计算例子
reference  = ['the', 'cat', 'sat', 'on', 'the', 'mat']
hypothesis = ['the', 'cat', 'sat', 'on', 'a',   'mat']
#                                           ^ 替换: the→a

# 手工统计
S = 1   # 'the' → 'a'
D = 0
I = 0
N = len(reference)
wer_manual = (S + D + I) / N

print(f'参考: {reference}')
print(f'假设: {hypothesis}')
print(f'S={S}, D={D}, I={I}, N={N}')
print(f'WER = ({S}+{D}+{I})/{N} = {wer_manual:.4f} = {wer_manual*100:.2f}%')

## 4. ✏️ 实现 `compute_wer(hypothesis, reference)`

**核心思路**：词级 Levenshtein 编辑距离

**DP 状态**：`dp[i][j]` = 将 `hypothesis[:i]` 变成 `reference[:j]` 的最少操作数

**转移方程**：
```
if hypothesis[i-1] == reference[j-1]:
    dp[i][j] = dp[i-1][j-1]           # 相同，不需要操作
else:
    dp[i][j] = 1 + min(
        dp[i-1][j],    # 删除 hypothesis[i-1]
        dp[i][j-1],    # 插入 reference[j-1]
        dp[i-1][j-1]   # 替换
    )
```

**返回值**：`edit_distance / len(reference)`

**验收标准**：

| 输入 | 期望输出 |
|---|---|
| `hyp=['the','cat'], ref=['the','cat']` | `0.0` |
| `hyp=['the'], ref=['the','cat']` | `0.5` (1删除/2词) |
| `hyp=['the','dog'], ref=['the','cat']` | `0.5` (1替换/2词) |

**边界情况**：`reference` 为空时，若 `hypothesis` 也为空返回 `0.0`；否则返回 `float('inf')`（无法用 0 个词作参考）。与 L67 的 `wer()` 约定一致。

### 补充：为什么删除、插入、替换的代价都统一是 1？

细心的同学可能会问：上面的转移方程里，"替换""删除""插入"的代价都写成了 `1`。但直觉上，"weather" 错成 "whether"（只是两个字母顺序颠倒）感觉应该"离得很近"；而"weather" 错成一个完全不相关的词（比如 "banana"），感觉应该"差得很远"。为什么这两种情况在公式里的代价都算 1，一视同仁？

**这不是疏忽，而是"标准编辑距离"本身的定义。**

标准编辑距离（Levenshtein 距离）要回答的问题很单纯：**把一个序列变成另一个序列，最少需要几次基本操作（增/删/改）？** 它把每一次操作都当作等价的"一步"——就像数"从家到公司最少要经过几个路口"，不管每个路口宽窄、远近，都只算一步。

**为什么要这样简化？**

1. **避免主观争议**：如果要区分"替换代价"，就要先回答"weather 变成 whether 该扣多少分？变成 banana 又该扣多少分？"——按拼写算？按发音算？按语义算？不同标准会给出完全不同的答案，容易吵不完。统一代价 1 避免了这个坑。
2. **保持算法干净**：统一代价让编辑距离满足"距离"应有的数学性质（比如三角不等式），DP 递推式简单、正确性容易证明，这也是为什么 L67 能从零把它推导清楚。
3. **对 WER 这个具体任务足够用**：ASR 评测只关心"识别错了几个词"，不关心"错得离不离谱"——对语音助手的下游任务来说，"weather"被误识别成"whether"和被误识别成"banana"，都是**同一次替换错误**，都同样会让指令被误解。

**如果确实需要区分"错得离谱程度"呢？**
有，这叫**加权编辑距离（weighted edit distance）**——给不同的操作或不同的字符/词对设置不同代价（比如发音相近的词替换代价小一些）。但这是标准编辑距离之外的另一个话题，本课以及 L67 只实现最基础的**统一代价为 1 的标准版本**。

一句话总结：代价为 1 不是"我们选择偷懒"，而是"最少操作数"这个概念本身的定义——想衡量别的东西（比如"错得多离谱"），就需要换一把不同的、加权的尺子。

> **💡 学习顺序说明**：本练习采用先试后学策略——先尝试实现 DP，建立直觉；
> L67 将系统推导 Levenshtein 算法的数学基础。
> 如果现在卡住，可跳到 L67 学完再回来完成此实现，两种顺序均可。
>
> 边界条件规范（实现时必须处理）：
> - `reference` 为空且 `hypothesis` 亦为空 → 返回 `0.0`（两边都空，没有错误）
> - `reference` 为空但 `hypothesis` 非空 → 返回 `float('inf')`（无法用 0 个词作参考，错率视为无穷）

In [ ]:
def compute_wer(hypothesis: list, reference: list) -> float:
    """计算词错率 WER = (S+D+I) / len(reference)。

    参数：
      hypothesis: 模型输出的词列表
      reference:  正确文本的词列表
    
    返回：
      WER 值（0.0 表示完美，> 1.0 表示大量错误）

    实现思路（Levenshtein 编辑距离）：
      用动态规划，dp[i][j] = 把 hypothesis[:i] 变成 reference[:j] 的最少操作数
    
    填表步骤：
      1. 边界：dp[0][j] = j（插入 j 个词），dp[i][0] = i（删除 i 个词）
      2. 转移：如果词相同，dp[i][j] = dp[i-1][j-1]；
               否则，dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
      3. 返回：dp[H][R] / R，其中 H, R = len(hypothesis), len(reference)
    
    优化提示：使用滚动数组，只保留一行 dp 值，空间复杂度 O(R)
    """
    H, R = len(hypothesis), len(reference)
    
    # ✏️ TODO: 边界条件（R==0 时：H==0 返回 0.0，否则返回 float('inf')）

    # ✏️ TODO: 初始化 dp 数组（长度 R+1，表示把空序列变成 ref[:j] 的代价）

    # ✏️ TODO: 遍历 hypothesis 的每个词，更新 dp（滚动数组，O(R) 空间）

    # ✏️ TODO: return dp[R] / R
    raise NotImplementedError(
        "请实现 compute_wer：参考上方伪代码补全所有 TODO 后再运行此单元。"
        "如需完整指导，可先完成 L67 再回来实现。"
    )

### 辅助：Levenshtein DP 的理解与伪代码

上面的公式和转移方程可能看起来很抽象。让我们用一个具体的例子来理解"删除"、"插入"、"替换"到底是对谁的操作，以及如何填 DP 表。

**理解 DP 表的含义**：

`dp[i][j]` = 把 `hypothesis[:i]`（假设词列表的前 i 个词）变成 `reference[:j]`（参考词列表的前 j 个词）所需的**最少操作数**。

**手工例子：假设文本与参考文本的对齐**

```
hypothesis = ['the', 'dog', 'sat']
reference  = ['the', 'cat', 'sat']
```

我们要填一个 4×4 的 DP 表（包括空前缀，所以 (H+1) × (R+1) = 4 × 4）：

```
        ''   'the'  'cat'  'sat'
   ''    0     1      2     3        ← 从空到参考的前j个词：需要j个插入操作
'the'    1     0      1     2
'dog'    2     1      1     2
'sat'    3     2      2     1

   ↑     ↑
   从空  这些行：从假设的前i个词到空，需要i个删除操作
   到假设
```

**逐格填法**：

1. **初始化第一行和第一列**：
   - 第一行 `dp[0][j]`：从空序列变到 ref[:j]，需要 j 个**插入**操作（把 j 个词一个一个加上去）
   - 第一列 `dp[i][0]`：从 hyp[:i] 变到空序列，需要 i 个**删除**操作（把 i 个词一个一个删除）

2. **填 dp[1][1]**（把"the"变成"the"）：
   - `hypothesis[0] == reference[0]`（都是"the"），所以不需要操作
   - `dp[1][1] = dp[0][0] = 0`  ✅

3. **填 dp[1][2]**（把"the"变成["the", "cat"]）：
   - `hypothesis[0] = 'the'` 与 `reference[1] = 'cat'` 不同
   - 选择最省的办法：
     - 从 dp[0][2]=2（把空变成["the","cat"]需要2个插入）+ 1个操作（删除hyp[0]）= 3
     - 从 dp[1][1]=0（把"the"变成"the"需要0个操作）+ 1个操作（插入"cat"）= 1  ← **最小**
     - 从 dp[0][1]=1（把空变成"the"需要1个插入）+ 1个操作（替换hyp[0]为"cat"）= 2
   - `dp[1][2] = 1 + min(3-1, 0, 1) = 1 + 0 = 1`

4. **填 dp[2][2]**（把["the","dog"]变成["the","cat"]）：
   - `hypothesis[1] = 'dog'` 与 `reference[1] = 'cat'` 不同
   - 三选一：
     - dp[1][2]=1 + 1（删除hyp[1]）= 2
     - dp[2][1]=1 + 1（插入ref[1]）= 2
     - dp[1][1]=0 + 1（替换hyp[1]为ref[1]）= 1  ← **最小**
   - `dp[2][2] = 1`

...以此类推到 `dp[3][3] = 1`（三个词中只有中间一个不同）。

**最终 WER = dp[3][3] / 3 = 1/3 ≈ 0.333**

---

**伪代码（用于实现）**：

```python
def compute_wer(hypothesis, reference):
    H, R = len(hypothesis), len(reference)
    
    # 边界条件
    if R == 0:
        return 0.0 if H == 0 else float('inf')
    
    # 初始化：只需要保留一行（滚动数组优化）
    prev_row = list(range(R + 1))  # [0, 1, 2, ..., R]
    
    # 遍历假设的每个词
    for i in range(1, H + 1):
        curr_row = [i]  # 第一格：从hyp[:i]变到空需要i个删除
        
        # 遍历参考的每个词
        for j in range(1, R + 1):
            if hypothesis[i - 1] == reference[j - 1]:
                # 相同，直接继承
                curr_row.append(prev_row[j - 1])
            else:
                # 不同，取三种操作的最小值
                delete_cost = prev_row[j] + 1       # 删除hypothesis[i-1]
                insert_cost = curr_row[j - 1] + 1   # 插入reference[j-1]
                replace_cost = prev_row[j - 1] + 1  # 替换
                curr_row.append(min(delete_cost, insert_cost, replace_cost))
        
        prev_row = curr_row  # 当前行变成下一次的前一行
    
    # 最终答案在prev_row[-1]（即dp[H][R]）
    return prev_row[-1] / R
```

关键点：
- **滚动数组**：只保留上一行的 dp 值，当前行计算后替换它，节省空间从 O(H×R) 到 O(R)
- **操作的含义**：
  - 删除：`prev_row[j]` 表示"假设已经处理到第 i-1 个词，参考处理到第 j 个词"，现在删除假设的第 i 个词，不处理参考
  - 插入：`curr_row[j-1]` 表示"假设处理到第 i 个词，参考处理到第 j-1 个词"，现在向假设中插入参考的第 j 个词
  - 替换：`prev_row[j-1]` 表示"假设处理到第 i-1 个词，参考处理到第 j-1 个词"，现在把假设的第 i 个词替换成参考的第 j 个词

---

In [ ]:
try:
    # 检查 compute_wer
    assert abs(compute_wer(['the', 'cat'], ['the', 'cat']) - 0.0) < 0.001, '完全一致 WER 应为 0'
    assert abs(compute_wer([], ['a']) - 1.0) < 0.001, '全删除 WER 应为 1'
    assert abs(compute_wer(['the', 'cat', 'sat'], ['the', 'sat', 'cat']) - 2/3) < 0.001, \
        '顺序互换（2次替换）WER 应为 2/3'
    assert abs(compute_wer(['a', 'b', 'c', 'd'], ['b', 'c']) - 1.0) < 0.001, \
        '多余词导致插入错误'
    print('✅ compute_wer 全部检查通过')
except (NotImplementedError, TypeError) as _e:
    print(f"⚠️  compute_wer 尚未实现 — 完成 TODO 后重新运行：{_e}")

## 5. 参数实验：错误类型分析

模拟 10 句话的 ASR 转写结果，统计 S/D/I 分布，观察不同类型错误对 WER 的贡献。

**预期现象**：
- 背景噪声场景：**D（删除）** 占比高，弱读音节被吞掉
- 口音场景：**S（替换）** 占比高，音素混淆
- 语速极快场景：**D** 再次升高，词边界丢失

实验中修改 `pairs` 列表，替换成自己的转写结果，观察统计变化。

In [ ]:
# 模拟 10 句话的参考文本与 ASR 假设（手工输入或真实转写）
pairs = [
    # (hypothesis_words, reference_words)
    (['hello', 'world'],                    ['hello', 'world']),           # 完美
    (['the', 'cat', 'sat'],                 ['the', 'cat', 'sat', 'down']),# 删除
    (['i', 'love', 'music'],                ['i', 'love', 'music']),       # 完美
    (['this', 'is', 'a', 'test'],           ['this', 'is', 'test']),       # 插入
    (['speech', 'recognition'],             ['speech', 'recognition']),    # 完美
    (['the', 'whether', 'is', 'nice'],      ['the', 'weather', 'is', 'nice']),# 替换
    (['aurora', 'audio'],                   ['aurora', 'audio', 'ai']),    # 删除
    (['deep', 'learning', 'rocks'],         ['deep', 'learning', 'rocks']),# 完美
    (['transformer', 'model'],              ['transformer', 'models']),    # 替换
    (['end', 'to', 'end', 'system'],        ['end', 'to', 'end', 'system']),# 完美
]

try:
    wers = [compute_wer(h, r) for h, r in pairs]
    avg_wer = np.mean(wers)

    print('句子 WER 分布：')
    for i, (w, (h, r)) in enumerate(zip(wers, pairs), 1):
        status = '✅' if w == 0 else '❌'
        print(f'  句{i:2d} {status}  WER={w:.3f}  ref_len={len(r)}')
    print(f'\n平均 WER = {avg_wer:.4f} = {avg_wer*100:.2f}%')
except (NotImplementedError, TypeError):
    print('⚠️  compute_wer 尚未实现——完成上方 ✏️ 练习后再运行本参数实验')

## 本课收束

本节实现了 `compute_wer`，把词级编辑距离压成 `(S+D+I)/N` 的 WER；后面的训练与评测都会复用它。下一课（L67）将从零实现 Levenshtein 动态规划，给 WER 先打好数学地基。

`compute_wer` 依赖下一课的编辑距离 DP；如果你想先做 WER 练习，建议先把 L67 看完再回来。

## ✏️ 白板挑战：ASR 核心概念手推（目标 10 分钟）

盖上屏幕，纸上推导：

**问 1**：Whisper 的标准输入特征是什么？帧参数是多少？  
（80-dim log-Mel，n_fft=400, hop=160, sr=16000 → 每帧=10ms，1秒≈100帧）

**问 2**：`compute_wer(['the', 'dog', 'sat'], ['the', 'cat', 'sat'])` 是多少？  
（1次替换/3词=1/3≈0.333）

**问 3**：WER 能超过 1.0 吗？什么情况下会？  
（能；当假设词数远多于参考词数时，插入次数大，WER=（S+D+I)/N可能>1）

**问 4**：端到端 ASR 为什么比传统 HMM-GMM 更容易处理新语言？  
（不需要手工设计发音词典；只需音频+文本对，可利用多语言弱监督数据联合训练）

**问 5**：声学编码器输出 `(B, T, 512)` Tensor，解码器怎么知道从哪里"注意"？  
（Cross-attention：解码器 query × 编码器 key/value，自动学习时间对齐，无需手工强制对齐）

推导完成后运行下方格验证。

### 补充：Cross-attention 的直观理解

白板挑战的问5涉及"Cross-attention"，这是 Transformer 解码器的核心机制。如果你还没学过 Transformer，这里简单解释。

**问题背景**：
- 编码器输出形状 `(B, T_enc, 512)`：B 个音频，每个有 T_enc 帧（如 3000），每帧 512 维隐状态
- 解码器需要生成 T_dec 个词（如 20 个词），每步输出一个 token
- 问题：解码器的每一步应该"注意"编码器的哪些帧？

**Self-Attention vs Cross-Attention**：

**Self-Attention**（解码器的自注意力）：
- 解码器只看自己已生成的 token（前 k 个）
- 作用：理解"前文语义"（比如已生成"the dog"，现在预测第三个词时，考虑"主语是单数"，倾向于选择谓语的单数形式）

**Cross-Attention**（解码器 × 编码器）：
- 解码器查看编码器的**所有**音频帧
- 作用：自动对齐"我现在要生成哪个词，应该注意音频的哪一段"
- 无需手工标注词-帧对齐，模型自己学

**Cross-Attention 的计算**：

```
Q (Query)   = 解码器当前步的隐状态    形状 (B, T_dec, d_model)
K (Key)     = 编码器的隐状态          形状 (B, T_enc, d_model)
V (Value)   = 编码器的隐状态          形状 (B, T_enc, d_model)

Scores = Q @ K.T / sqrt(d_model)       形状 (B, T_dec, T_enc)
         ↑
         这是一个"相似度矩阵"，每行是一个解码步，每列是一个编码帧
         
Attention = Softmax(Scores, dim=-1)    形状 (B, T_dec, T_enc)
            (把每一行按照编码帧方向归一化)
         ↑
         这样，对于解码步 t，每个编码帧 s 都有一个权重
         
Output = Attention @ V                 形状 (B, T_dec, d_model)
         (对编码帧加权平均)
```

**直观理解**：
- **Q·K.T** 计算"解码当前词"与"编码某帧"的相似度
- **Softmax** 把相似度转成归一化权重（全和为1）
- **权重 × V** 对编码帧的隐状态进行加权平均

例如，如果解码器在生成"cat"，可能会给编码帧 500-700（对应音频中"喵"的部分）分配大权重，给其他帧分配小权重。

**为什么除以 sqrt(d_model)？**
- 当 d_model 很大时，Q·K 的值会很大，导致 Softmax 的梯度很小（梯度消失）
- 除以 sqrt(d_model) 让分数的方差保持在 1 左右，稳定训练（这叫**Scaled Dot-Product Attention**）

---

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：Whisper 特征形状（不依赖真实音频）
sr, n_fft, hop, n_mels = 16000, 400, 160, 80
duration_s = 30.0  # Whisper 固定 30 秒窗口
# Whisper 先补零/居中再分帧，帧数 = 样本数 / hop（若用无补零公式 1+(N-n_fft)//hop 会得 2998）
n_frames_30s = int(sr * duration_s) // hop
assert n_frames_30s == 3000, f"30s帧数={n_frames_30s}"
print(f"Q1 ✅  Whisper输入: 80-dim log-Mel, 30s → {n_frames_30s}帧 (hop=10ms)")

# 问2：WER计算验证
def _ref_wer(hyp, ref):
    if not ref: return 0.0
    H, R = len(hyp), len(ref)
    dp = [[0]*(R+1) for _ in range(H+1)]
    for i in range(H+1): dp[i][0] = i
    for j in range(R+1): dp[0][j] = j
    for i in range(1, H+1):
        for j in range(1, R+1):
            if hyp[i-1] == ref[j-1]: dp[i][j] = dp[i-1][j-1]
            else: dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[H][R] / R

wer_q = _ref_wer(['the','dog','sat'], ['the','cat','sat'])
assert abs(wer_q - 1/3) < 1e-9, f"wer={wer_q}"
print(f"Q2 ✅  WER(['the','dog','sat'], ['the','cat','sat']) = {wer_q:.4f} = 1/3")

# 问3：WER > 1.0 验证
wer_over1 = _ref_wer(['a','b','c','d','e'], ['x'])   # 4插入+1替换=5，/1=5.0
assert wer_over1 > 1.0
print(f"Q3 ✅  WER可超1.0: hyp=['a','b','c','d','e'], ref=['x'] → WER={wer_over1:.1f}")

# 问4：端到端WER概念验证（用参考实现跑几对）
test_pairs = [
    (['hello', 'world'], ['hello', 'world'], 0.0),
    (['hello'],          ['hello', 'world'], 0.5),
    (['hi', 'world'],   ['hello', 'world'], 0.5),
]
for hyp, ref, expected in test_pairs:
    got = _ref_wer(hyp, ref)
    assert abs(got - expected) < 1e-9, f"hyp={hyp},ref={ref}: {got}≠{expected}"
print(f"Q4 ✅  端到端ASR评估: 3组WER计算验证通过 (0.0/0.5/0.5)")

# 问5：Cross-attention形状验证
#
# Cross-attention 的工作流：
#   1. 解码器当前词的隐状态 Q: (B, 1, d_model) ← 每步只有1个位置（当前输出位置）
#   2. 编码器全部帧的隐状态 K, V: (B, T_enc, d_model) ← 所有音频帧都可用
#   3. 计算相似度：Q @ K.T: (B, 1, T_enc) ← "当前词对每个编码帧的注意力权重"
#   4. Softmax 和加权平均：
#      attention_weights = Softmax(scores / sqrt(d_model), dim=-1)  [B, 1, T_enc]
#      context = attention_weights @ V  [B, 1, d_model]
#
# 在实践中，解码器生成 T_dec 个词，所以会有 T_dec 次这样的计算，
# 全部串起来就是 (B, T_dec, T_enc) 的完整 attention scores。

B, T_enc, T_dec, d_model = 2, 3000, 20, 512
print(f"\n→ Q5 Cross-attention 详解:")
print(f"  编码器输出 (B, T_enc, d_model) = ({B}, {T_enc}, {d_model})")
print(f"  解码器每步输出 (B, 1, d_model) = ({B}, 1, {d_model})")
print(f"  Cross-attention scores: ({B}, {T_dec}, {T_enc})")
print(f"    ↑ 对于每一个解码步，我们计算它对编码器所有 {T_enc} 帧的注意力")

# 演示单步 cross-attention
Q_single = np.random.randn(B, 1, d_model)
K = np.random.randn(B, T_enc, d_model)
V = np.random.randn(B, T_enc, d_model)
scores_single = (Q_single @ K.transpose(0, 2, 1)) / np.sqrt(d_model)
assert scores_single.shape == (B, 1, T_enc), f"单步形状错误 {scores_single.shape}"
print(f"  ✓ 单步 scores 形状: {scores_single.shape}")

# 演示完整 cross-attention（T_dec 步）
Q_full = np.random.randn(B, T_dec, d_model)
scores_full = (Q_full @ K.transpose(0, 2, 1)) / np.sqrt(d_model)
assert scores_full.shape == (B, T_dec, T_enc), f"完整形状错误 {scores_full.shape}"
print(f"  ✓ 完整 scores 形状: {scores_full.shape}")

# Softmax 和值函数
attention = scores_full  # Softmax 会沿着 T_enc 维度（第-1维）归一化
attention_weights = np.ones((B, T_dec, T_enc)) / T_enc  # 简化版（实际应该 softmax）
context = attention_weights @ V
assert context.shape == (B, T_dec, d_model)
print(f"  ✓ Attention加权后: {context.shape}")

print(f"\n🎉 Cross-attention 通过！解码器通过这个机制自动学会对齐音频和文字")
print("   本课结束。L67 会从头实现 Levenshtein DP，建立 WER 数学基础。")

In [ ]:
# ✏️ 本课自评
l66_review = {
    "e2e_vs_hmm_understood":   None,  # 理解端到端vs传统HMM-GMM的关键区别？True/False
    "wer_formula_memorized":   None,  # 记住WER=(S+D+I)/N，能手算简单例子？True/False
    "compute_wer_impl":        None,  # compute_wer()DP实现正确，3个断言通过？True/False
    "whisper_features":        None,  # 记住Whisper输入=80-dim log-Mel，30s→3000帧？True/False
    "whiteboard_passed":       None,  # 白板挑战5道全部完成？True/False
}

unfilled = [k for k, v in l66_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l66_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L66 全部通关！进入 L67：Edit Distance 从零实现')

---

→ **下一课**　[L67 · Edit Distance 从零实现](L67_edit_distance.ipynb)

> 下节课将学习 **Edit Distance 从零实现**：Levenshtein 动态规划，WER 的数学基础。